# 04 · Mode shape vs DC bias and load

Reconstructs the mode-shape map at a list of (bias, load) conditions, and uses the
**bias dependence to separate the piezoresponse and electrostatic channels** — so the
D-ESBS comes out without switching PPLN domains.

The detected response is
```
    Z(x, f; V) = Z_piezo(x, f) + (V - V_cpd) · S_elec(x, f)
```
because the first-harmonic electrostatic force goes as (V_dc − V_cpd)·V_ac while the
piezoresponse does not depend on bias. A complex straight-line fit in V at each (x, f)
splits them: the **slope is the pure electrostatic channel** (exact — no V_cpd needed,
so the D-ESBS is rigorous) and the **intercept is the response at V = 0**, which still
carries a −V_cpd·S_elec term unless you supply `V_CPD`.

**Acquisition is organised per position, not per condition.** Moving the laser,
AutoWedge, InvOLS and engaging cost about a minute; changing bias costs a second. So
each position is visited once and every condition is measured there. That is several
times faster and — since the bias fit is per position — removes position-repeatability
error from the comparison.

Everything is checkpointed after each position, and resumes.

**Windows / Igor only.** Same setup as notebook 03: fixed tune window, no auto-recenter,
laser parked where `START_AT` says.

## 1 · Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import win32com.client
from activemodemap.asylum import AFMLaserSweepAutomation, AsylumInstrument
from activemodemap import (make_conditions, run_series, reconstruct_series,
                           separate_channels, estimate_v_cpd, channel_spots,
                           load_checkpoint, plot_state)

igor = win32com.client.Dispatch('IgorPro.Application')
print('Connected to Igor Pro')

## 2 · Parameters

In [ ]:
file_loc      = r'D:\User Data\Liam\ActiveModeMap\Series1'   # must exist
base_filename = 'AMapS'
os.makedirs(file_loc, exist_ok=True)

# --- geometry / reachable span (x = 0 is the clamped BASE) ---
PROBE_L_UM = 225.0
START_AT   = 'free_end'
REACH_UM   = 118.0
step_um    = 1.0
x_grid     = np.arange(PROBE_L_UM - REACH_UM, PROBE_L_UM + 1e-6, step_um)
HW_SIGN_TOWARD_FREE_END = +1

# --- conditions -----------------------------------------------------------
# At least 3 biases are needed for a testable linear fit (2 define a line exactly
# and leave no residual). Keep |V| below the coercive voltage or you will switch
# domains and the response stops being linear in bias -- the residual will tell you.
BIAS_V  = [-2.0, -1.0, 0.0, 1.0, 2.0]
LOAD_NN = [1000.0]                 # add more for a load series, e.g. [500, 1000, 2000]
conditions = make_conditions(BIAS_V, LOAD_NN, vary='bias_inner')

REF_INDEX = len(conditions) // 2   # the condition that drives position selection
V_CPD     = 0.0                    # set from KPFM if known; see estimate_v_cpd below

# --- reconstruction -------------------------------------------------------
DNS_BAND_HZ   = (330e3, 470e3)     # where to look for the resonance/null
RANK          = 4
MIN_POSITIONS = 6                  # = RANK + 2
MAX_POSITIONS = 14
DNS_CI_TOL_UM = 1.0
RECALIBRATE_EACH = True

CHECKPOINT = os.path.join(file_loc, 'series_checkpoint.npz')

n = len(conditions)
print(f'{n} conditions x ~{MAX_POSITIONS} positions')
print(f'~{n} tunes per position; laser move + calibrate + engage happens ONCE per position')
print('conditions:', [str(c) for c in conditions])

## 3 · Instrument, direction check

In [ ]:
automation = AFMLaserSweepAutomation(igor, file_loc, base_filename, log_filename=None)
automation.eigenmode_center_freq = float(np.mean(DNS_BAND_HZ))
automation.autowedge_pause = 15.0
automation.invols_bounds = (4e-8, 10e-7)
automation.resonance_band_Hz = DNS_BAND_HZ

inst = AsylumInstrument(automation, load_nN=LOAD_NN[0], dc_bias_V=BIAS_V[0],
                        recalibrate_each=RECALIBRATE_EACH,
                        analysis_band_Hz=None,          # full spectrum to the fit
                        start_at=START_AT, span_um=PROBE_L_UM,
                        x_limits_um=(x_grid[0], x_grid[-1]),
                        hw_sign_toward_free_end=HW_SIGN_TOWARD_FREE_END)

# Walking inward from the free end every move should read "toward base".
inst.preview_moves(list(x_grid[::-20]))

## 4 · Run the series

Interrupt any time — the checkpoint holds every completed position, and re-running
this cell resumes from it rather than repeating measurements.

In [ ]:
series = run_series(inst, x_grid, conditions,
                    ref_index=REF_INDEX, rank=RANK,
                    min_positions=MIN_POSITIONS, max_positions=MAX_POSITIONS,
                    dns_band_Hz=DNS_BAND_HZ, dns_ci_tol_um=DNS_CI_TOL_UM,
                    start_near_um=(x_grid[-1] if START_AT == 'free_end' else x_grid[0]),
                    checkpoint_path=CHECKPOINT, resume=True, verbose=True)
inst.close()
print(f"\n{series['x_um'].size} positions x {len(conditions)} conditions "
      f"x {series['freq_Hz'].size} frequencies")

## 5 · Reconstruct every condition

In [ ]:
recs = reconstruct_series(series, x_grid, rank=RANK,
                          auto_band=True, band_halfwidth_Hz=70e3)

# auto_band re-centres the search on each condition's own resonance. That matters
# for a load series: contact stiffening moves the resonance by tens of kHz, and a
# band fixed at one load silently misses the peak at another.
ok = [r for r in recs if r is not None]
V  = np.array([r['condition'].bias_V for r in ok])
Ld = np.array([r['condition'].load_nN for r in ok])
fr = np.array([r['f_res_Hz'] for r in ok])
dn = np.array([r['dns_um'] for r in ok])

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for L_ in np.unique(Ld):
    s = Ld == L_
    ax[0].plot(V[s], fr[s] / 1e3, 'o-', label=f'{L_:.0f} nN')
    ax[1].plot(V[s], PROBE_L_UM - dn[s], 'o-', label=f'{L_:.0f} nN')
ax[0].set_xlabel('DC bias (V)'); ax[0].set_ylabel('resonance (kHz)')
ax[1].set_xlabel('DC bias (V)'); ax[1].set_ylabel('D-NS (µm from free end)')
for a in ax[:2]:
    a.legend(fontsize=8, frameon=False)
if np.unique(Ld).size > 1:
    for v in np.unique(V):
        s = V == v
        ax[2].plot(Ld[s], PROBE_L_UM - dn[s], 'o-', label=f'{v:+.1f} V')
    ax[2].set_xlabel('load (nN)'); ax[2].set_ylabel('D-NS (µm from free end)')
    ax[2].legend(fontsize=8, frameon=False)
else:
    ax[2].axis('off')
plt.tight_layout()
fig.savefig(os.path.join(file_loc, 'Series_vs_condition.png'), dpi=150, bbox_inches='tight')

## 6 · Channel separation — D-NS and D-ESBS from the bias dependence

`rel_resid` is the linearity check. A response that is genuinely linear in bias gives
a residual at the noise level. A large residual means something else is going on —
domain switching at high |V|, electrostriction, or the contact drifting during the
sweep — and the separation should not be trusted.

In [ ]:
LOAD_FOR_CHANNELS = LOAD_NN[0]

ch = separate_channels(series, load_nN=LOAD_FOR_CHANNELS, v_cpd=V_CPD)
print(f"linearity: peak residual {100*ch['rel_resid']:.1f}% of peak |Z|")

# V_cpd proxy: the bias minimising the total response, per position. It equals V_cpd
# only where the piezoresponse vanishes, so read it near the D-NS and treat it as a
# rough correction, not a calibrated KPFM number.
vcpd_per_pos, vcpd_med = estimate_v_cpd(ch, band_Hz=DNS_BAND_HZ)
print(f'V_cpd proxy: median {vcpd_med:+.3f} V across positions')

spots = channel_spots(ch, x_grid, rank=RANK, band_Hz=DNS_BAND_HZ)
print(f"\nD-NS   = {PROBE_L_UM - spots['dns']['value_um']:.2f} µm from the free end")
print(f"D-ESBS = {PROBE_L_UM - spots['desbs']['value_um']:.2f} µm from the free end")
print('\nNote the two use different estimators on purpose: the D-NS is the '
      'antiresonance branch crossing, the D-ESBS a spatial null of the '
      'electrostatic channel at resonance. The branch crossing cannot '
      'distinguish the channels — it gives the same position for both.')

In [ ]:
# the two channel maps side by side
f_k = ch['freq_Hz'] / 1e3
ext = [x_grid[0], x_grid[-1], f_k[0], f_k[-1]]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, key, ttl, spot in ((ax[0], 'dns', 'piezoresponse channel (intercept)', 'dns'),
                          (ax[1], 'desbs', 'electrostatic channel (dZ/dV)', 'desbs')):
    M = np.abs(spots[key]['Zrec']).T
    a.imshow(np.log10(M + 1e-14), origin='lower', aspect='auto', extent=ext, cmap='viridis')
    for xv in series['x_um']:
        a.axvline(xv, color='w', lw=0.5, alpha=0.7)
    a.axvline(spots[spot]['value_um'], color='#e34948', ls='--', lw=1.5)
    a.set_title(ttl, fontsize=10); a.set_xlabel('position (µm)'); a.set_ylabel('frequency (kHz)')
ax[2].plot(ch['x_um'], vcpd_per_pos, 'o-', color='#2a78d6', ms=4)
ax[2].axvline(spots['dns']['value_um'], color='#e34948', ls='--', label='D-NS')
ax[2].axvline(spots['desbs']['value_um'], color='#2f9e5f', ls=':', label='D-ESBS')
ax[2].set_xlabel('position (µm)'); ax[2].set_ylabel('V_cpd proxy (V)')
ax[2].legend(fontsize=8, frameon=False); ax[2].set_title('bias minimising |Z|', fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(file_loc, 'Series_channels.png'), dpi=150, bbox_inches='tight')

np.savez(os.path.join(file_loc, 'Series_result.npz'),
         x_um=series['x_um'], freq_Hz=series['freq_Hz'], Z=series['Z'],
         x_grid=x_grid, bias_V=ch['bias_V'], load_nN=ch['load_nN'],
         piezo=ch['piezo'], elec=ch['elec'], resid=ch['resid'], v_cpd=V_CPD,
         dns_um=spots['dns']['value_um'], desbs_um=spots['desbs']['value_um'],
         probe_L_um=PROBE_L_UM)
print('saved Series_result.npz, Series_channels.png, Series_vs_condition.png')

## Notes

- **Signal-to-noise sets whether this works.** The D-NS comes from the antiresonance
  notch, ~2 % of the resonance peak deep. What matters is noise relative to the
  off-resonance baseline; in simulation the recovery held from 2 % to 50 % of
  baseline. If the map shows many branch crossings scattered along the beam, the
  notch is unresolvable — average longer or narrow the tune window. Smoothing does
  not help: wide enough to suppress the wiggles also erases the notch.
- **Keep |V| below the coercive voltage.** Switching domains mid-sweep breaks the
  linear-in-bias assumption; `rel_resid` is the diagnostic.
- **Load changes re-engage by default** (`set_load(reengage=True)`). A large setpoint
  jump while in contact drives the tip into the surface.
- The load series needs `auto_band=True`: the contact resonance stiffens with load.
- Only the reference condition drives position selection, so all conditions share a
  position set and are directly comparable.